In [1156]:
# 설치
# !pip install -qU pypdf

In [1157]:
# OCR
# !pip install -qU rapidocr-onnxruntime

In [1158]:
# from langchain_community.document_loaders.parsers import RapidOCRBlobParser

In [1159]:
# !pip install pdfminer.six

In [1160]:
# !pip install langchain-community pymupdf

In [1161]:
# !pip install -qU docx2txt

In [1162]:
# !pip install unstructured python-pptx

In [1163]:
# import nltk
# nltk.download('punkt_tab')

In [1164]:
# Reference
# https://m.blog.naver.com/htk1019/223442628204

In [1165]:
import bs4

In [1166]:
from langchain_core.documents import Document

In [1167]:
from langchain_community.document_loaders.parsers import RapidOCRBlobParser

In [1168]:
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader, PDFMinerLoader, UnstructuredPDFLoader
from langchain_community.document_loaders import Docx2txtLoader, UnstructuredPowerPointLoader, TextLoader
# from langchain_community.document_loaders import WebBaseLoader

In [1169]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

In [1170]:
from langchain_experimental.text_splitter import SemanticChunker

In [1171]:
#!pip install langchain_experimental

In [1172]:
from langchain_huggingface import HuggingFaceEmbeddings

In [1173]:
from IPython.display import HTML

In [1174]:
import re
import json
import warnings
warnings.filterwarnings("ignore")

In [1175]:
# 1. 문서 리스트 준비
sentences = [
    "우리나라는 2022년에 코로나가 유행했다.",
    "우리나라 2024년 GDP 전망은 3.0%이다.",
    "우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.",
    "LangChain is a robust framework built to orchestrate LLM applications.",
    "FAISS (Facebook AI Similarity Search) is a highly efficient library for clustering and similarity search of dense vectors.",
    "Chroma is a highly efficient, open-source vector database tailored for AI applications.",
    "Hugging Face is a massive platform hosting thousands of open-source machine learning models."
]

In [1176]:
all_raw_text = []

In [1177]:
def transform_clean_string(to_replace): 
    """remove unnecessary characters"""

    if isinstance(to_replace, (str)):
        to_replace = to_replace.strip()
        to_replace = re.sub(r'(?<!\.)(\n|\r\n)', ' ', to_replace)
        to_replace = re.sub(r'\t|\\t', ' ', to_replace)
        to_replace = re.sub(r' +', ' ', to_replace)

    # merge all texts
    all_raw_text.append(to_replace)
    
    return to_replace

In [1178]:
# --
# PDF Loader
# --
filepath = './files/asiabrief_3-26.pdf'

loader = PyPDFLoader(filepath, mode="page", images_inner_format="markdown-img", images_parser=RapidOCRBlobParser())
# loader = PyPDFDirectoryLoader("data/")
# loader = PDFMinerLoader(pdf_filepath)
# loader = UnstructuredPDFLoader(pdf_filepath, mode="elements")

# data = [Document(page_content=transform_trim_string(text.page_content), metadata=text.metadata) for i, text in enumerate(loader.load())]
data = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in loader.load()]

In [1179]:
# --
# DOC Loader
# --
filepath = './files/Sample.docx'

# all_raw_text = []

# loader = Docx2txtLoader(filepath)
# data = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in loader.load()]

In [1180]:
# --
# DOC Loader
# --
filepath = './files/test.log'

# all_raw_text = []

# loader = TextLoader(filepath, encoding="utf-8")
# data = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in loader.load()]

In [1181]:
print(f'총 분할된 도큐먼트 수: {len(data)}')

총 분할된 도큐먼트 수: 5


In [1182]:
# data

In [1183]:
# print(f"전체 텍스트 추출: {''.join(all_raw_text)}")

In [1184]:
# --
# Hugging Face가 인터넷을 검색하지 않도록 오프라인 환경 변수 설정
# os.environ["HF_HUB_OFFLINE"] = "1"
# os.environ["TRANSFORMERS_OFFLINE"] = "1"

# 로컬에 저장된 모델 경로 불러오기
model_path = "../huggingfase_model/distiluse-base-multilingual-cased-v1/"

# model = SentenceTransformer(model_path)
# # 3. Calculate embeddings by calling model.encode()
# embeddings = model.encode(sentences, convert_to_numpy=True)
# print(f"Embeddings shape: {embeddings.shape}")

embedding_model = HuggingFaceEmbeddings(
    model_name=model_path
)
print(f"Embeddings: {embedding_model}")
# --

Embeddings: model_name='../huggingfase_model/distiluse-base-multilingual-cased-v1/' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [1185]:
# Usually set to 10% to 20% of your total chunk_size
# 2. Split documents into small, manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=300)
# text_splitter = CharacterTextSplitter(chunk_size=3000, chunk_overlap=500, separator = '\n\n')
# SemanticChunker 초기화 (백분위수 기반 분할)
# text_splitter = SemanticChunker(embedding_model, breakpoint_threshold_type="percentile") # percent, standard_deviation, interquartile 등 선택 가능)
docs = text_splitter.split_documents(data)

In [1186]:
print(f'TextSplitter 후, 총 분할된 도큐먼트 수: {len(docs)}')

TextSplitter 후, 총 분할된 도큐먼트 수: 5


In [1187]:
# The metadata corresponding to each text
metadata_list = [
    {"source": "docs-intro.txt", "author": "LangChain Team"}
]
# Create document objects from your sentence list
documents = text_splitter.create_documents(sentences, metadata_list*7)
documents

[Document(metadata={'source': 'docs-intro.txt', 'author': 'LangChain Team'}, page_content='우리나라는 2022년에 코로나가 유행했다.'),
 Document(metadata={'source': 'docs-intro.txt', 'author': 'LangChain Team'}, page_content='우리나라 2024년 GDP 전망은 3.0%이다.'),
 Document(metadata={'source': 'docs-intro.txt', 'author': 'LangChain Team'}, page_content='우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.'),
 Document(metadata={'source': 'docs-intro.txt', 'author': 'LangChain Team'}, page_content='LangChain is a robust framework built to orchestrate LLM applications.'),
 Document(metadata={'source': 'docs-intro.txt', 'author': 'LangChain Team'}, page_content='FAISS (Facebook AI Similarity Search) is a highly efficient library for clustering and similarity search of dense vectors.'),
 Document(metadata={'source': 'docs-intro.txt', 'author': 'LangChain Team'}, page_content='Chroma is a highly efficient, open-source vector database tailored for AI applications.'),
 Document(metadata={'source': 'docs-intro.txt', 'author': 'LangCha

In [1188]:
# docs

In [1189]:
# -- Directory

In [1190]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

# 확장자별 로더 매핑
loader_mapping = {
    '.log': TextLoader,
    '.pdf': PyPDFLoader,
}

# 매핑된 로더를 loader_cls에 지정
# glob='./*.*'
# loader = DirectoryLoader('./files', glob="**/*.pdf", loader_cls=PyPDFLoader)
# loader = DirectoryLoader(path='./files', glob=['**/*.txt', '**/*.log'])
# docs = loader.load_and_split()

# 1. 각 확장자별로 DirectoryLoader 생성
txt_loader1 = DirectoryLoader('./files', glob='**/*.txt', loader_cls=TextLoader)
txt_loader2 = DirectoryLoader('./files', glob='**/*.log', loader_cls=TextLoader)
pdf_loader = DirectoryLoader('./files', glob='**/*.pdf', loader_cls=PyPDFLoader)

# 2. 문서 로드
txt_docs1 = txt_loader1.load()
txt_docs2 = txt_loader2.load()
pdf_docs = pdf_loader.load()

# 3. 문서 리스트 병합
docs = txt_docs1 + txt_docs2 + pdf_docs

In [1191]:
print(f"총 {len(docs)}개의 문서를 불러왔습니다.")

총 6개의 문서를 불러왔습니다.


In [1192]:
for s in docs:
    print(s.metadata.get("source"))

files/test.log
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf


In [1193]:
docs = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in docs]

In [1194]:
# docs

In [1195]:
docs[0].page_content

'24/10/11 12:35:22 INFO myLogger: 2024-10-11 12:35:22.075 DB Execute for TableName : Test 24/10/11 12:35:38 ERROR Executor: Exception in task 1.98 in stage 0.0 (TID 107) java.lang.OutOfMemoryError: Java heap space'

In [1196]:
HTML('<pre style="background-color: #A52A2A; color: white;">Job is being performed correctly</pre>')